# Concurso Docente UBA - Extracción Automática desde cv_es.yaml

Este notebook genera texto base para copiar/pegar en el nuevo sistema de concursos docentes.

Uso recomendado:
1. Ejecutar la celda 2 (setup).
2. Ejecutar cada celda de sección (a, b, c, ...).
3. Copiar el resultado y ajustar redacción final si hace falta.

In [ ]:
import yaml
from pathlib import Path
from datetime import date

DATA_PATH = Path("../cv_db/cv_es.yaml")

with DATA_PATH.open("r", encoding="utf-8") as f:
    cv_data = yaml.safe_load(f)

print(f"Datos cargados desde: {DATA_PATH.resolve()}")

def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def list_to_text(value):
    out = []
    for item in as_list(value):
        if isinstance(item, dict):
            for k, v in item.items():
                out.append(f"{k}: {v}")
        else:
            out.append(str(item))
    return [x.strip() for x in out if str(x).strip()]

def get_path(*keys, default=None):
    node = cv_data
    for key in keys:
        if not isinstance(node, dict) or key not in node:
            return [] if default is None else default
        node = node[key]
    return node

def print_title(title):
    bar = "=" * len(title)
    print(bar)
    print(title)
    print(bar)

def format_date(d):
    return str(d or "").strip()

def is_ongoing(date_str):
    return "actual" in str(date_str).lower()

today = date.today().isoformat()
print(f"Fecha de ejecución: {today}")

## a. TITULOS UNIVERSITARIOS OBTENIDOS

In [ ]:
print_title("a) TÍTULOS UNIVERSITARIOS OBTENIDOS")

titulos = get_path("Educación", "Educación Universitaria")
for i, t in enumerate(as_list(titulos), 1):
    nombre = t.get("name", "")
    institucion = t.get("location", "")
    periodo = format_date(t.get("date", ""))
    detalles = "; ".join(list_to_text(t.get("description")))

    print(f"{i}. {nombre}")
    print(f"   Facultad/Universidad: {institucion}")
    print(f"   Período: {periodo}")
    if detalles:
        print(f"   Detalles: {detalles}")
    print()

## b. ANTECEDENTES DOCENTES E ÍNDOLE DE LAS TAREAS

In [ ]:
print_title("b) ANTECEDENTES DOCENTES E ÍNDOLE DE LAS TAREAS")

docencia = get_path("Experiencia", "Docencia y Formación")

def naturaleza_designacion(nombre):
    low = str(nombre).lower()
    if "invitado" in low:
        return "Invitado"
    if "facilitador" in low:
        return "Facilitador"
    if "co-organizador" in low or "organizador" in low:
        return "Organizador / Docente"
    return "Regular"

def contains_supervision(nombre, descripcion_list):
    """Check if item is about supervision (to optionally filter out)"""
    low_name = str(nombre).lower()
    supervision_keywords = ("supervisión", "supervision", "mentor")
    if any(k in low_name for k in supervision_keywords):
        return True
    for desc in as_list(descripcion_list):
        if isinstance(desc, dict):
            continue
        if any(k in str(desc).lower() for k in supervision_keywords):
            return True
    return False

# Sección 1: Docencia formal (excluyendo supervisión pura)
print("DOCENCIA Y ACTIVIDADES FORMATIVAS:")
for i, item in enumerate(as_list(docencia), 1):
    nombre = item.get("name", "")
    periodo = format_date(item.get("date", ""))
    institucion = item.get("location", "")
    designacion = naturaleza_designacion(nombre)
    tareas = list_to_text(item.get("description"))

    # Opcional: saltar supervisiones puras
    if contains_supervision(nombre, tareas):
        continue

    print(f"{i}. {nombre}")
    print(f"   Institución/lugar: {institucion}")
    print(f"   Período: {periodo}")
    print(f"   Naturaleza de la designación: {designacion}")
    print("   Tareas desarrolladas:")
    for t in tareas:
        print(f"   - {t}")
    print()

# Sección 2: Materiales publicados en Zenodo
print("\n" + "=" * 60)
print("MATERIALES DOCENTES Y RECURSOS PUBLICADOS")
print("=" * 60)

zenodo_items = []

# Buscar Zenodo en Docencia y Formación
for item in as_list(docencia):
    detalles = list_to_text(item.get("description"))
    for det in detalles:
        if "zenodo" in det.lower():
            zenodo_items.append({
                "source": "Docencia",
                "title": item.get("name", ""),
                "date": item.get("date", ""),
                "location": item.get("location", ""),
                "material": det
            })

# Buscar Zenodo en Posters y Presentaciones Orales
posters = get_path("Producción", "Posters y Presentaciones Orales")
for item in as_list(posters):
    detalles = list_to_text(item.get("description"))
    for det in detalles:
        if "zenodo" in det.lower():
            zenodo_items.append({
                "source": "Presentación",
                "title": item.get("title", ""),
                "event": item.get("event", ""),
                "date": item.get("date", ""),
                "material": det
            })

if zenodo_items:
    for i, zitem in enumerate(zenodo_items, 1):
        if "title" in zitem:
            print(f"\n{i}. {zitem['title']}")
            if "date" in zitem and zitem['date']:
                print(f"   Año: {zitem['date']}")
            if "location" in zitem and zitem['location']:
                print(f"   Institución: {zitem['location']}")
            if "event" in zitem and zitem['event']:
                print(f"   Evento: {zitem['event']}")
            print(f"   Material: {zitem['material']}")
else:
    print("No se encontraron materiales en Zenodo en la CV actual.")

print("\n" + "=" * 60)
print("(Puedes copiar cualquiera de estas dos secciones o ambas)")
print("=" * 60)


## c. ANTECEDENTES CIENTÍFICOS Y PUBLICACIONES

In [ ]:
print_title("c) ANTECEDENTES CIENTÍFICOS")

print("PUBLICACIONES:")
publicaciones = get_path("Producción", "Publicaciones")
for i, p in enumerate(as_list(publicaciones), 1):
    autores = p.get("authors", "")
    fecha = format_date(p.get("date", ""))
    titulo = p.get("title", "")
    revista = p.get("journal", "")
    detalles = "; ".join(list_to_text(p.get("description")))

    print(f"{i}. {autores} ({fecha}). {titulo}. {revista}.")
    if detalles:
        print(f"   Detalles: {detalles}")

print()
print("OTROS ANTECEDENTES RELACIONADOS CON LA ESPECIALIDAD (INVESTIGACIÓN):")
investigacion = get_path("Experiencia", "Investigación")
for i, item in enumerate(as_list(investigacion), 1):
    nombre = item.get("name", "")
    fecha = format_date(item.get("date", ""))
    lugar = item.get("location", "")
    desc = "; ".join(list_to_text(item.get("description")))
    print(f"{i}. {nombre}. {lugar}. Período: {fecha}.")
    if desc:
        print(f"   Detalles: {desc}")
    print()

## d. CURSOS, CONFERENCIAS Y TRABAJOS DE INVESTIGACIÓN

In [ ]:
print_title("d) CURSOS DE ESPECIALIZACIÓN, CONFERENCIAS Y TRABAJOS")

cursos = get_path("Cursos y Congresos")
for i, c in enumerate(as_list(cursos), 1):
    nombre = c.get("name", "")
    fecha = format_date(c.get("date", ""))
    lugar = c.get("location", "")
    duracion = c.get("extension", "")
    idioma = c.get("language", "")
    desc = "; ".join(list_to_text(c.get("description")))

    print(f"{i}. {nombre}.")
    print(f"   Lapso: {fecha}")
    print(f"   Lugar: {lugar}")
    if duracion:
        print(f"   Duración/carga horaria: {duracion}")
    if idioma:
        print(f"   Idioma: {idioma}")
    if desc:
        print(f"   Actividad: {desc}")
    print()

print("TRABAJOS DE INVESTIGACIÓN (EDITOS / PREPRINTS):")
for i, p in enumerate(as_list(get_path("Producción", "Publicaciones")), 1):
    journal = str(p.get("journal", "")).lower()
    if "biorxiv" in journal or "arxiv" in journal:
        print(f"- {p.get('title','')} ({p.get('date','')}) - {p.get('journal','')}")

## e. PARTICIPACIÓN EN CONGRESOS O ACONTECIMIENTOS SIMILARES

In [ ]:
print_title("e) PARTICIPACIÓN EN CONGRESOS O ACONTECIMIENTOS SIMILARES")

posters = get_path("Producción", "Posters y Presentaciones Orales")
for i, item in enumerate(as_list(posters), 1):
    titulo = item.get("title", "")
    evento = item.get("event", "")
    lugar = item.get("location", "")
    fecha = format_date(item.get("date", ""))
    autores = item.get("authors", "")
    calidad = "; ".join(list_to_text(item.get("description")))

    print(f"{i}. {evento}.")
    print(f"   Título/actividad: {titulo}")
    print(f"   Lugar: {lugar}")
    print(f"   Lapso: {fecha}")
    if autores:
        print(f"   Autores/representación: {autores}")
    if calidad:
        print(f"   Calidad de participación: {calidad}")
    print()

## f. ACTUACIÓN EN INSTITUCIONES Y CARGOS EN SECTOR PÚBLICO/PRIVADO

In [ ]:
print_title("f.1) ACTUACIÓN EN UNIVERSIDADES E INSTITUTOS")

# Palabras clave para filtrar: bioimage analyst, postdoc con beca, freelance
claves_bioimage = ("bioimage", "bioimágenes", "analista")
claves_beca = ("beca", "funded", "subsidiado", "vetenskapsrådet", "seal of excellence", "sello de excelencia")

print("ACTIVIDADES COMO ANALISTA DE BIOIMÁGENES EN ARGENTINA Y SUECIA:")
idx = 1

# Sección Académico: buscar "Analista de bioimágenes"
for item in as_list(get_path("Experiencia", "Académico")):
    nombre = item.get("name", "")
    if any(k in nombre.lower() for k in claves_bioimage):
        fecha = format_date(item.get("date", ""))
        lugar = item.get("location", "")
        desc = "; ".join(list_to_text(item.get("description")))
        print(f"{idx}. {nombre}")
        print(f"   Organismo/entidad: {lugar}")
        print(f"   Lapso: {fecha}")
        if desc:
            print(f"   Tareas: {desc}")
        print()
        idx += 1

# Sección Investigación: buscar postdocs con financiamiento en Sweden/instituciones relevantes
print("POSTDOCTORADOS Y BECAS DE INVESTIGACIÓN:")
for item in as_list(get_path("Experiencia", "Investigación")):
    nombre = item.get("name", "")
    lugar = item.get("location", "")
    desc_list = list_to_text(item.get("description"))
    
    # Detectar si tiene financiamiento mencionado o si está en Sweden/institución relevante
    tiene_beca = any(k in " ".join(desc_list).lower() for k in claves_beca)
    en_suecia = "suecia" in lugar.lower() or "sweden" in lugar.lower()
    
    if tiene_beca or en_suecia:
        fecha = format_date(item.get("date", ""))
        print(f"{idx}. {nombre}")
        print(f"   Institución/lugar: {lugar}")
        print(f"   Lapso: {fecha}")
        if desc_list:
            for d in desc_list:
                print(f"   - {d}")
        print()
        idx += 1

# Sección Profesionales: buscar trabajo freelance en análisis de bioimágenes
print("=" * 60)
print("ACTIVIDADES PROFESIONALES COMO ANALISTA (Consultoría/Freelance):")
for i, item in enumerate(as_list(get_path("Experiencia", "Profesionales")), 1):
    nombre = item.get("name", "")
    fecha = format_date(item.get("date", ""))
    lugar = item.get("location", "")
    desc = "; ".join(list_to_text(item.get("description")))
    print(f"{i}. {nombre}")
    print(f"   Entidad/lugar: {lugar}")
    print(f"   Lapso: {fecha}")
    if desc:
        print(f"   Funciones: {desc}")
    print()


## g. FORMACIÓN DE RECURSOS HUMANOS

In [ ]:
print_title("g) FORMACIÓN DE RECURSOS HUMANOS")

docencia = as_list(get_path("Experiencia", "Docencia y Formación"))
claves_supervision = ("supervisión", "supervision", "mentor", "estudiante")
claves_becas = ("beca", "funded", "subsidiado", "seal of excellence", "sello de excelencia")

# Sección 1: Supervisión de estudiantes
print("SUPERVISIÓN Y FORMACIÓN DE ESTUDIANTES:")
idx_supervision = 1
for item in docencia:
    nombre = str(item.get("name", "")).lower()
    descripcion = list_to_text(item.get("description"))
    texto_desc = " ".join(descripcion).lower()
    
    # Detectar si es un item de supervisión
    is_supervision = any(c in nombre for c in claves_supervision) or any(c in texto_desc for c in claves_supervision)
    
    if is_supervision:
        print(f"{idx_supervision}. {item.get('name','')}")
        print(f"   Institución/lugar: {item.get('location','')}")
        print(f"   Lapso: {item.get('date','')}")
        if descripcion:
            for d in descripcion:
                print(f"   - {d}")
        print()
        idx_supervision += 1

if idx_supervision == 1:
    print("(No se encontraron supervisiones de estudiantes)")
    print()



## h. SÍNTESIS DE APORTES ORIGINALES

Durante el período 2016-2024 desarrollé aportes originales en la interfaz entre biofísica, microscopía avanzada y análisis cuantitativo de bioimágenes, en instituciones de Argentina y Suecia. Mi trabajo se orientó a vincular modelado físico, adquisición de señales e inferencia cuantitativa para extraer información biológica a partir de la conversión de datos de imagen en cantitdades fotofísicas y el modelado de los sensores biológicos subyacentes.

En mi etapa doctoral (2016-2021, CONICET - Universidad de Buenos Aires) trabajé en la cuantificación de procesos dinámicos en sistemas biológicos complejos, integrando modelado físico y análisis computacional para describir fenómenos de organización celular y tisular. Allí consolidé una base conceptual y técnica en instrumentación, procesamiento de señales e interpretación cuantitativa de imágenes.

Durante mi etapa posdoctoral (2022-2024, Karolinska Institutet, Suecia) profundicé en el desarrollo y validación de metodologías reproducibles para análisis de bioimágenes, incorporando herramientas de ciencia de datos e inteligencia artificial para relacionar señales ópticas con variables biológicas medibles.

Mis aportes originales pueden sintetizarse en tres ejes:

- Integración de fundamentos físicos, modelado y análisis computacional para estudiar sistemas biológicos dinámicos.
- Desarrollo de metodologías reproducibles para extraer variables biológicas cuantitativas a partir de señales de microscopía.
- Aplicación de estas estrategias en contextos interdisciplinarios e internacionales de investigación.

## i. SÍNTESIS DE ACTUACIÓN PROFESIONAL Y/O EXTENSIÓN UNIVERSITARIA

Mi actuación profesional reciente se desarrolló principalmente entre 2024 y 2025 en BioImage Informatics Facility (Science for Life Laboratory, Uppsala Universitet y National Bioinformatics Infrastructure Sweden), donde trabajé como analista de bioimágenes para proyectos de investigación de distintas áreas biológicas. En ese ámbito diseñé y optimicé flujos de análisis para múltiples modalidades de microscopía, articulando la pregunta biológica con la teoría fotofísica de la señal, el análisis cuantitativo y la evaluación estadística.

Este trabajo no solo involucró el procesamiento de imágenes, sino también la mejora del diseño experimental: selección de condiciones de adquisición, definición de controles, resolución temporal y espacial, estrategia de muestreo y formulación de modelos para describir la dinámica del sistema en estudio. El objetivo fue producir análisis capaces de contrastar hipótesis biológicas de manera cuantitativa y orientar decisiones experimentales posteriores.

En extensión universitaria y transferencia de conocimiento, participé en el dictado de cursos y talleres intensivos sobre análisis de bioimágenes, microscopía e inteligencia artificial aplicada a la biología en instituciones de Suecia y América Latina. Asimismo, contribuí a la producción y circulación de recursos educativos abiertos, incluyendo materiales docentes, tutoriales y código, para facilitar la adopción de metodologías reproducibles por parte de la comunidad científica.

También formo parte de GloBIAS prácticamente desde sus inicios, coordinando actividades de presentación de la sociedad ante la comunidad científica, relevamiento del estado del campo a nivel mundial y organización de actividades educativas. En paralelo, participo en Latin America BioImaging (LABI), articulando acciones con GloBIAS para fortalecer las capacidades regionales en bioimágenes en América Latina.

## j. OTROS ELEMENTOS DE JUICIO VALIOSOS

Los elementos adicionales que considero más valiosos son aquellos que reflejan reconocimiento externo de mi trayectoria y capacidad de formación en contextos competitivos e internacionales.

- Becas y distinciones obtenidas en Argentina y Suecia entre 2016 y 2024, incluyendo la beca doctoral CONICET, un financiamiento posdoctoral de Vetenskapsrådet y el reconocimiento Marie Skłodowska-Curie Actions Seal of Excellence.
- Invitacones para preparar y dictar cursos y talleres en universidades e institutos de Ecuador, Chile, Uruguay, Suecia y Argentina, como indicador de reconocimiento académico y capacidad de transferencia.
- Convocatoria como jurado de dos tesis de licenciatura en la Universidad de Buenos Aires, expresión de confianza institucional en mi criterio académico.

## k. PLAN DE LABOR DOCENTE, INVESTIGACIÓN Y EXTENSIÓN

##### **Sus puntos de vista sobre temas básicos de su campo del conocimiento que deben transmitirse a los alumnos; la importancia relativa y ubicación de su área en el currículo de la carrera. Medios que propone para mantener actualizada la enseñanza y para llevar a la práctica los cambios que sugiere.**

En el marco de mi postulación como **Profesor Adjunto** en la carrera de Física, mi plan docente se centrará en la sinergia entre el modelado de la dinámica del sistema bajo estudio, la comprensión del rol de sus parámetros y el análisis de datos necesario para evaluar estadísticamente si ese modelo describe adecuadamente el fenómeno observado. Considero que este enfoque permite transmitir a los estudiantes una visión de la física como disciplina capaz de formular, contrastar y refinar descripciones cuantitativas de sistemas complejos.

Mi experiencia en **física interdisciplinaria**, en particular en **biofísica**, orienta esta propuesta. En mi área de trabajo, los conocimientos de física permiten traducir datos provenientes de detectores y sensores de cámaras en variables fotofísicas y, a partir de su interpretación, en variables de interés biológico. Por eso, entiendo que esta perspectiva debe ocupar un lugar relevante dentro de la formación, especialmente en la articulación entre óptica, laboratorio, métodos numéricos, análisis y ciencia de datos y problemas contemporáneos de investigación.

Para mantener actualizada la enseñanza y llevar estos cambios a la práctica, en los laboratorios de enseñanza básicos haré especial hincapié en el versionado de datos, el seguimiento de cambios en los análisis y la reproducibilidad de los experimentos. Entre las actividades propuestas, promoveré que los grupos compartan entre sí la descripción metodológica de sus experiencias, de modo que deban explicitar con claridad sus procedimientos para que otros los reproduzcan, identificar qué información es importante incluir en la descripción experimental y mejorar sus diseños al incorporar otras perspectivas. Asimismo, buscaré que los estudiantes aprendan a simular los experimentos a realizar y a comparar esas simulaciones con los observables experimentales. En este contexto, el auge de los modelos fundacionales de visión y de los agentes de inteligencia artificial ha transformado el análisis de bioimágenes; por eso no solo me mantengo actualizado en estos avances, sino que también enseño su uso crítico, junto con herramientas de versionado de datos y de análisis para sostener prácticas de trabajo reproducibles.